# Comparing Granularity [Step 5 - Which Strategy, and What It Costs]

> **MLCourse - Agentic AI - Advanced RAG - Contextual Retrieval**

Four strategies for the small-to-big problem, one comparison:

1. **fixed chunks** - the baseline, one compromise size
2. **parent document** - index children, return parents
3. **sentence window** - index sentences, return match plus neighbours
4. **static headers** - index chunk-plus-context, return the chunk

We measure retrieval quality with keyword-based relevance rules, and context
size in characters, because context size is the cost you pay at generation time.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# "Parent" units: paragraphs. Big enough to answer from, too big to retrieve
# precisely. These are the documents we will later cut into small children.
parents = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("parent paragraphs:", len(parents))
print("mean parent length:", int(sum(len(p) for p in parents) / len(parents)), "chars")

parent paragraphs: 237
mean parent length: 379 chars


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_index(texts):
    """Embed a list of texts and return the normalised matrix."""
    return encoder.encode(texts, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=False)


def search(index, texts, query, top_n=5):
    """Return [(position, score)] of the best matches in `index`."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sims = index @ q
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("encoder ready:", encoder.get_sentence_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoder ready: 384 dimensions


### 2. Ground truth

Relevance is decided mechanically: a returned context is relevant to a question
if it contains all the keywords of at least one of that question's keyword
groups. Same rule for every strategy, defined before the experiment - the same
discipline as
[`../11_reranking/04_measuring_the_lift.ipynb`](../11_reranking/04_measuring_the_lift.ipynb).

Note the subtlety this experiment introduces: strategies that return **more
text** are more likely to satisfy a keyword rule. That is not cheating, it is
the actual effect we are studying - but it means we must report context size
alongside precision, or the comparison is meaningless.

In [5]:
EVAL_QUESTIONS = [
    {"q": "Why was the White Rabbit in such a hurry?",
     "any_of": [["rabbit", "hurry"], ["rabbit", "late"], ["oh dear", "late"]]},
    {"q": "What happened when Alice drank from the little bottle?",
     "any_of": [["drink", "bottle"], ["bottle", "telescope"], ["drank", "telescope"]]},
    {"q": "What game does the Queen of Hearts make everyone play?",
     "any_of": [["croquet"], ["flamingo", "hedgehog"]]},
    {"q": "Who does Alice meet at the mad tea party?",
     "any_of": [["hatter", "dormouse"], ["march hare", "hatter"], ["tea", "dormouse"]]},
    {"q": "What advice does the Caterpillar give Alice?",
     "any_of": [["caterpillar", "mushroom"], ["caterpillar", "keep your temper"],
                ["caterpillar", "who are you"]]},
    {"q": "How does the Cheshire Cat disappear?",
     "any_of": [["grin", "vanish"], ["cheshire cat", "grin"], ["vanished", "grin"]]},
    {"q": "What does the Queen shout whenever she is angry?",
     "any_of": [["off with"], ["queen", "executed"]]},
    {"q": "What happens at the trial of the Knave of Hearts?",
     "any_of": [["knave", "tarts"], ["jury", "verdict"], ["sentence", "verdict"]]},
]


def text_is_relevant(text, question):
    low = text.lower()
    return any(all(w in low for w in group) for group in question["any_of"])


print(len(EVAL_QUESTIONS), "evaluation questions")

8 evaluation questions


### 3. Build all four retrievers

Each is a function `question -> list of context strings`, so they are directly
interchangeable.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

full_text = "\n\n".join(parents)

# --- 1. fixed chunks -------------------------------------------------------
fixed_texts = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=80).split_text(full_text)
fixed_vectors = build_index(fixed_texts)

# --- 2. parent document ----------------------------------------------------
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
child_texts, child_parent = [], []
for pid, parent in enumerate(parents):
    for piece in child_splitter.split_text(parent):
        child_texts.append(piece)
        child_parent.append(pid)
child_vectors = build_index(child_texts)

# --- 3. sentence window ----------------------------------------------------
SENT_RE = re.compile(r"(?<=[.!?])\s+")
sentences = [s.strip() for s in SENT_RE.split(full_text) if len(s.strip()) > 25]
sentence_vectors = build_index(sentences)

# --- 4. static contextual headers -----------------------------------------
CHAPTER_RE = re.compile(r"^CHAPTER\s+([IVXL]+)\.?\s*(.*)$", re.IGNORECASE)
chapter_of_parent, current = {}, "Front matter"
for pid, parent in enumerate(parents):
    m = CHAPTER_RE.match(parent.strip())
    if m:
        current = f"Chapter {m.group(1)} - {m.group(2).strip()[:40]}"
    chapter_of_parent[pid] = current

hdr_chunks, hdr_parent = [], []
for pid, parent in enumerate(parents):
    for piece in RecursiveCharacterTextSplitter(
            chunk_size=600, chunk_overlap=80).split_text(parent):
        hdr_chunks.append(piece)
        hdr_parent.append(pid)
hdr_texts = [f"Alice's Adventures in Wonderland. {chapter_of_parent[hdr_parent[i]]}.\n---\n{c}"
             for i, c in enumerate(hdr_chunks)]
hdr_vectors = build_index(hdr_texts)

print(f"fixed chunks   : {len(fixed_texts)}")
print(f"child chunks   : {len(child_texts)} over {len(parents)} parents")
print(f"sentences      : {len(sentences)}")
print(f"headed chunks  : {len(hdr_texts)}")

fixed chunks   : 217
child chunks   : 622 over 237 parents
sentences      : 484
headed chunks  : 267


In [7]:
TOP_K = 3


def r_fixed(question):
    return [fixed_texts[p] for p, _ in search(fixed_vectors, fixed_texts, question, TOP_K)]


def r_parent(question):
    hits = search(child_vectors, child_texts, question, top_n=TOP_K * 4)
    out, seen = [], set()
    for pos, _ in hits:
        pid = child_parent[pos]
        if pid in seen:
            continue
        seen.add(pid)
        out.append(parents[pid])
        if len(out) == TOP_K:
            break
    return out


def r_window(question, window=3):
    hits = search(sentence_vectors, sentences, question, top_n=TOP_K * 4)
    out, used = [], set()
    for pos, _ in hits:
        if any(abs(pos - u) <= window for u in used):
            continue
        used.add(pos)
        out.append(" ".join(sentences[max(0, pos - window):pos + window + 1]))
        if len(out) == TOP_K:
            break
    return out


def r_headers(question):
    return [hdr_chunks[p] for p, _ in search(hdr_vectors, hdr_texts, question, TOP_K)]


STRATEGIES = {"fixed chunks": r_fixed, "parent document": r_parent,
              "sentence window": r_window, "static headers": r_headers}
print("strategies:", list(STRATEGIES))

strategies: ['fixed chunks', 'parent document', 'sentence window', 'static headers']


### 4. Run the comparison

Three metrics per strategy:

- **precision@3** - what fraction of the three returned contexts are relevant.
- **hit rate** - did *any* of the three contain the answer. This is the metric
  that matters most for RAG: the LLM only needs the answer to be present once.
- **mean context size** - the price paid, in characters.

In [8]:
results = {}
for name, fn in STRATEGIES.items():
    precisions, hits, sizes = [], [], []
    for question in EVAL_QUESTIONS:
        ctxs = fn(question["q"])
        flags = [text_is_relevant(c, question) for c in ctxs]
        precisions.append(sum(flags) / max(len(flags), 1))
        hits.append(1.0 if any(flags) else 0.0)
        sizes.append(sum(len(c) for c in ctxs))
    results[name] = {
        "p@3": float(np.mean(precisions)),
        "hit": float(np.mean(hits)),
        "chars": float(np.mean(sizes)),
        "per_q": precisions,
    }

print(f"{'strategy':<18}{'P@3':>8}{'hit rate':>11}{'mean chars':>13}"
      f"{'hit per 1k chars':>19}")
print("-" * 70)
for name, r in results.items():
    efficiency = r["hit"] / (r["chars"] / 1000)
    print(f"{name:<18}{r['p@3']:>8.3f}{r['hit']:>11.3f}{r['chars']:>13.0f}"
          f"{efficiency:>19.3f}")

strategy               P@3   hit rate   mean chars   hit per 1k chars
----------------------------------------------------------------------
fixed chunks         0.375      0.750         1244              0.603
parent document      0.292      0.750         1350              0.555
sentence window      0.417      1.000         4224              0.237
static headers       0.333      0.625         1224              0.510


The final column is the one people forget. Returning more text almost always
raises the hit rate - in the limit, returning the whole book has a hit rate of
1.0 and is useless. **Hit rate per 1000 characters** asks the real question: how
much answer are you getting per unit of context window spent?

In [9]:
print("per-question precision@3\n")
header = f"{'question':<40}" + "".join(f"{n[:13]:>15}" for n in STRATEGIES)
print(header)
print("-" * len(header))
for i, question in enumerate(EVAL_QUESTIONS):
    row = "".join(f"{results[n]['per_q'][i]:>15.2f}" for n in STRATEGIES)
    print(f"{question['q'][:38]:<40}{row}")

per-question precision@3

question                                   fixed chunks  parent docume  sentence wind  static header
----------------------------------------------------------------------------------------------------
Why was the White Rabbit in such a hur             0.67           0.33           0.67           0.67
What happened when Alice drank from th             0.33           0.33           0.33           0.67
What game does the Queen of Hearts mak             0.00           0.33           0.67           0.00
Who does Alice meet at the mad tea par             0.67           0.00           0.33           0.00
What advice does the Caterpillar give              0.67           0.33           0.33           0.67
How does the Cheshire Cat disappear?               0.33           0.00           0.33           0.33
What does the Queen shout whenever she             0.00           0.67           0.33           0.00
What happens at the trial of the Knave             0.33          

### 5. Same question, four answers

Metrics are a proxy. Read the actual generations.

In [10]:
QUESTION = "What advice does the Caterpillar give Alice?"

for name, fn in STRATEGIES.items():
    ctxs = fn(QUESTION)
    context = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(ctxs))
    answer = ask(
        "Answer the question using ONLY the context below. If the context is "
        "insufficient, say exactly what is missing.\n\n"
        f"Context:\n{context}\n\nQuestion: {QUESTION}\nAnswer:"
    )
    print("=" * 74)
    print(f"{name}  ({len(context)} chars of context)")
    print("=" * 74)
    print(answer)
    print()
    time.sleep(2.0)          # pace the free-tier token budget

fixed chunks  (1381 chars of context)
Based on the context provided, the Caterpillar advises Alice that "One side will make you grow taller, and the other side will make you grow shorter."



parent document  (1056 chars of context)
Based on the context provided, the Caterpillar advises Alice that "One side will make you grow taller, and the other side will make you grow shorter."



sentence window  (3975 chars of context)
Based on the context provided, the Caterpillar gives Alice the following advice: “One side will make you grow taller, and the other side will make you grow shorter.”



static headers  (910 chars of context)
The Caterpillar advises Alice that “One side will make you grow taller, and the other side will make you grow shorter.”



### 6. Choosing

| your corpus | start with |
|---|---|
| flowing prose, transcripts, books | **sentence window** |
| documents with real structure - sections, tickets, pages | **parent document** |
| chunks that lose their subject when split | **contextual headers** |
| short self-contained records (FAQs, product entries) | **fixed chunks** - you do not have this problem |

And the composition that a strong production system actually uses:

```
   parent document retriever          <- decouple retrieval from generation
     with contextual headers on children  <- fix orphan children
     -> rerank the CHILDREN               <- ../11_reranking (short text, fits the window)
     -> return the PARENTS to the LLM     <- full context for answering
```

Reranking children rather than parents is the detail worth remembering: children
are short enough to survive the cross-encoder's 512-token limit, and you expand
to parents only after the ordering is settled.

### 7. Ask the model to summarise the tradeoff


In [11]:
table = "\n".join(
    f"{name}: precision@3={r['p@3']:.3f}, hit_rate={r['hit']:.3f}, "
    f"mean_context_chars={r['chars']:.0f}"
    for name, r in results.items()
)

print(table)
print("\n--- recommendation ---")
print(ask(
    "You are a RAG engineer. Below are measured results for four chunking / "
    "retrieval strategies on a 237-paragraph literary corpus with 8 evaluation "
    "questions. In 5-7 sentences, say which strategy gives the best answer "
    "quality per unit of context window spent, and which you would ship for a "
    "corpus of long narrative prose. Note explicitly if a strategy's higher hit "
    "rate is simply the result of returning more text.\n\n" + table
))

fixed chunks: precision@3=0.375, hit_rate=0.750, mean_context_chars=1244
parent document: precision@3=0.292, hit_rate=0.750, mean_context_chars=1350
sentence window: precision@3=0.417, hit_rate=1.000, mean_context_chars=4224
static headers: precision@3=0.333, hit_rate=0.625, mean_context_chars=1224

--- recommendation ---


Sentence window retrieval offers the best answer quality per unit of context window spent, achieving the highest precision@3 (0.417) and a perfect hit rate (1.000). However, its superior hit rate is explicitly the result of returning significantly more text, as its mean context length of 4,224 characters is roughly three times that of the other strategies. In contrast, fixed chunks and static headers provide similar, lower precision scores with much tighter context windows, making them more efficient but less comprehensive. Parent document retrieval performs poorly on precision (0.292) despite a high hit rate, indicating it retrieves relevant sections but dilutes them with excessive surrounding noise. For a corpus of long narrative prose, I would ship fixed chunks because they balance reasonable precision with minimal context overhead, avoiding the token bloat of sentence windows. This approach ensures the model focuses on the most relevant narrative segments without being overwhelmed 

### 8. Key takeaways

- Always report **context size** next to retrieval quality; a strategy that
  returns more text will score better on any content-based metric.
- **Hit rate** matters more than precision for RAG - the answer only needs to be
  present once.
- Sentence windows suit prose; parent documents suit structured corpora;
  contextual headers fix chunks that lost their subject.
- These strategies **compose**, and they compose with reranking - rerank the
  small units, return the big ones.

That closes this module. Next:
[`../14_graph_rag`](../14_graph_rag/README.md), which asks what to do when the
answer is not in any single chunk at all.